# 03 — Results  [RESULTS]

The final model, its fit, and its substantive interpretation. Nothing exploratory or inconclusive lives here — see `04_robustness_and_open_issues.ipynb` for everything else.

## Contents

- **3.1** Final NB model on `rate_social`
- **3.2** Predicted rates at representative predictor values
- **3.3** Model fit and diagnostics
- **3.4** Spatial distribution of the outcome
- **3.5** Substantive interpretation

### 3.1 Final NB model on `rate_social`  [RESULTS — SETTLED]

`racialised_reviews ~ foreign_population_pct_c + log_population_density_c + sgb_ii_recipients_per_1k_c + is_east + in_split_city`, `offset=log(social_reviews)`, cluster-robust SEs by state.

| predictor | rate ratio | 95% CI | p |
|---|---|---|---|
| Intercept | 0.015 | [0.014, 0.015] | <0.001 |
| foreign_population_pct_c | 1.041 | [1.025, 1.058] | <0.001 |
| log_population_density_c | 1.087 | [1.003, 1.178] | 0.042 |
| sgb_ii_recipients_per_1k_c | 1.006 | [1.004, 1.008] | <0.001 |
| is_east | 1.258 | [0.982, 1.613] | 0.070 |
| in_split_city | 0.757 | [0.597, 0.961] | 0.022 |

*(Source: `outputs/B9_main_model_rate_ratios.csv`, `archive/supervisor_run_2026-08-16.ipynb` cell `B9`.)*

### 3.2 Predicted rates at representative predictor values  [RESULTS — SETTLED]

Holding other predictors at their mean and `is_east`/`in_split_city` at 0, moving `foreign_population_pct` from its 25th to 75th percentile shifts the predicted `rate_social` from **1.23%** to **1.65%** — a ~34% relative increase across the interquartile range of the strongest predictor. (Computed directly from the 3.1 model's fitted coefficients; the equivalent p25/p75 comparison for `rate_total` is in `rate_total_analysis.ipynb` C6: 0.48% → 0.71%.)

### 3.3 Model fit and diagnostics  [RESULTS — SETTLED]

- **Dispersion (Pearson χ²/df):** 0.826 — well-specified NB fit (α̂=0.375).
- **VIF:** all five predictors well below the conventional concern threshold of 10 (max 4.61, `log_population_density_c`).

| variable | VIF |
|---|---|
| foreign_population_pct_c | 3.59 |
| log_population_density_c | 4.61 |
| sgb_ii_recipients_per_1k_c | 1.96 |
| is_east | 1.50 |
| in_split_city | 1.55 |

- **Influence (Cook's distance), top 5:** Offenburg (0.655), Lörrach – Müllheim (0.176), Duisburg II (0.137), Dahme-Spreewald – Teltow-Fläming III (0.071), Chemnitz (0.062). Offenburg and Lörrach – Müllheim are the two constituencies already flagged by name in `redo_socio_political_relations.ipynb`'s influence check; the recurring **~0.037 outlier visible across all nine LOWESS panels (§2.5) is Essen III** (`rate_social`=0.0402, Cook's d=0.057, 6th-largest) — none of the top-5 by Cook's distance are dropped from the reported model; this is a description of where leverage concentrates, not a robustness failure (§4.2's B10 five-subset check already confirms coefficients are stable to excluding these).
*(Source: `outputs/B10_vif.csv`, `outputs/B10_cooks_distance_top10.csv` — this influence table already existed in `archive/supervisor_run_2026-08-16.ipynb`, verified here by independent recomputation against `nb_main`; not a new run.)*

**Caveat found during the 2026-08-24/25 gap-closing pass:** the 0.826 dispersion figure above uses a moment-matched (plug-in) α=0.375. `analysis_10072026/presentation_18082026.md` R4.4 asks for the version that estimates α jointly by MLE instead — its output (`outputs/deck/T7_specification_ladder.csv`) reports α=0.1854513, fitting notably better (AIC 2196.6 vs. 2241.7) but reading as **overdispersed** (1.502) rather than underdispersed. Independently reproduced with `statsmodels.discrete.discrete_model.NegativeBinomial` (joint MLE) — α, AIC, and dispersion all matched to full precision. Coefficients, signs, and every significance call are essentially unchanged between the two estimators (max rate-ratio difference 0.003) — 3.1's results are robust to this choice, but "well-specified, dispersion 0.826" is itself estimator-dependent and shouldn't be stated as if it weren't.

### 3.4 Spatial distribution of the outcome  [RESULTS — SETTLED]

**Global Moran's I = 0.2814** (999 permutations, p=0.001) on `rate_social` across all 299 constituencies — moderate, highly significant positive spatial autocorrelation. LISA identifies two coherent High-High hotspot clusters (Ruhr-area NRW; Berlin) plus a smaller Baden-Württemberg pair. Full spatial-dependence accounting, including how this compares to the *model's residual* Moran's I, is consolidated in §4.6. *(Source: `outputs/B7_lisa_counts.csv`, `archive/supervisor_run_2026-08-16.ipynb` cell `B7`.)*

### 3.5 Substantive interpretation  [RESULTS — SETTLED]

Racialisation is not evenly distributed: it rises with foreign-population share and population density, is elevated in the East (though only marginally, p=0.070, cluster-robust), and is *lower* in split cities — a genuine, moderate, spatially structured pattern (§3.4), not an artifact of any single predictor or venue (cross-checked five ways in §4.2's B10 robustness table). The model explains real structure (dispersion 0.826, all VIFs low) but does not fully absorb the spatial clustering present in the raw outcome — see §4.6 for exactly how much survives in the residuals, and §4.5 for what this model cannot speak to (who is racialising whom).